# DANN-CORAL Cost Model

Domain adaptation cost model with SAINT encoder, CORAL alignment, and a DANN GRL domain head.

## Architecture Overview

```text
               Input
                 │
                 ▼
         SAINT Encoder (Gf)
                 │ z
     ┌───────────┼────────────┐
     ▼           ▼            ▼
Regression     Domain       CORAL
  Head (Gy)   Classifier     Loss
     │           │            │
     ▼           ▼            ▼
Cost Pred   Domain Label   Cov Align
                 ▲
                 │
                GRL
```

## 1. Install Dependencies

In [ ]:
!pip install -q torch>=2.0 scikit-learn pandas numpy scipy lightgbm


## 2. Imports

In [ ]:

import torch
import torch.nn as nn
from torch.autograd import Function
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import random
import time
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import hashlib
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from pathlib import Path
import warnings
import json
import os
import matplotlib.pyplot as plt




## 3. Data Preprocessing

In [ ]:

# Columns to log-scale (large numeric values)
# Columns to log-scale (large, positive-skew numeric values)
_LOG_SCALE_CANDIDATES = [
    # Memory / buffer sizes (both legacy + underscore variants)
    "buffer_pool_size", "cache_effective_size", "cache_effective_size_",
    "per_query_memory", "maintenance_memory", "temp_memory",
    "log_buffer_size", "log_capacity", "temp_file_limit_",
    
    # Resource / capacity knobs that can span wide ranges
    "max_connections_", "io_parallelism",
    
    # Workload counters
    "blks_hit", "blks_read",
    "disk_read_bytes", "disk_read_count",
    "disk_write_bytes", "disk_write_count",
    "tup_fetched", "tup_returned", "tup_inserted", "tup_updated", "tup_deleted",
    "xact_commit", "xact_rollback",
    "conflicts",
    
    # Optional timing/size-like knobs (keep if present)
    "commit_delay_",
    "vacuum_cost_limit_", "vacuum_cost_page_dirty_", "vacuum_cost_page_hit_", "vacuum_cost_page_miss_",
]

LABEL_COL = "cost"
DOMAIN_COL = "domain_id"
QP_EMB_COL = "qp_emb_vector"
SOURCE_DOMAIN = 3
META_COLS = ["db_engine", "hardware", "ram_gb"]


def get_log_scale_cols(df):
    return [c for c in _LOG_SCALE_CANDIDATES if c in df.columns]


def get_mask_cols(df):
    return [c for c in df.columns if c.startswith("mask_")]


def get_feature_cols(df, mask_cols):
    exclude = set(mask_cols + [LABEL_COL, DOMAIN_COL, QP_EMB_COL] + META_COLS)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return [c for c in numeric_cols if c not in exclude]

def expand_qp_emb(df, col=None):
    """Expand qp_emb_vector list column into per-dimension float columns"""
    if col is None:
        col = QP_EMB_COL
    emb_matrix = np.vstack(df[col].values)
    qp_cols = [f"qp_emb_{i}" for i in range(emb_matrix.shape[1])]
    emb_df = pd.DataFrame(emb_matrix, columns=qp_cols, index=df.index)
    return emb_df, qp_cols


def preprocess(da, normalization_mode="per_engine", log_target=False):
    """
    Complete preprocessing pipeline:
    1. Convert embeddings from JSON strings
    2. Log-scale large numeric features
    3. Fill NaN values
    4. Filter zero-variance features
    5. Expand query plan embeddings
    6. Fit categorical encoders and feature scaler
    7. NORMALIZE COSTS with selected mode: per_engine or global
    
    Returns: processed df, feature columns, encoders, scaler, and normalization params
    """
    df = da.copy()

    # Convert qp_emb_vector from JSON strings if needed
    if df[QP_EMB_COL].dtype == object and isinstance(df[QP_EMB_COL].iloc[0], str):
        df[QP_EMB_COL] = df[QP_EMB_COL].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
        print("INFO: Converted qp_emb_vector from JSON strings")

    log_scale_cols = get_log_scale_cols(df)
    if log_scale_cols:
        print(f"INFO: Log-scaling {len(log_scale_cols)} feature columns")
    # Log-scale large numeric features
    for col in log_scale_cols:
        if col in df.columns:
            df[col] = np.log1p(df[col].clip(lower=0))

    mask_cols = get_mask_cols(df)
    feat_cols = get_feature_cols(df, mask_cols)

    # Replace non-finite feature/mask values, then fill NaNs
    df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    df[mask_cols] = df[mask_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # Filter zero-variance features
    feat_cols_filtered = [c for c in feat_cols if df[c].var() > 1e-10]
    dropped_zero_var = [c for c in feat_cols if c not in feat_cols_filtered]
    if dropped_zero_var:
        print(f"WARN: Dropped {len(dropped_zero_var)} zero-variance features: {dropped_zero_var[:5]}")

    # Clip extreme values to prevent scaling instability
    for col in feat_cols_filtered:
        df[col] = np.clip(df[col], -1e4, 1e4)

    # Expand query plan embeddings and concatenate
    emb_df, qp_cols = expand_qp_emb(df)
    emb_df = emb_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    df = pd.concat([df.reset_index(drop=True), emb_df.reset_index(drop=True)], axis=1)
    print(f"INFO: Expanded {len(qp_cols)} query plan embedding dimensions")

    # Clean cost column BEFORE computing normalization stats
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    bad_cost_mask = ~np.isfinite(df[LABEL_COL].to_numpy())
    bad_cost_count = int(bad_cost_mask.sum())
    if bad_cost_count > 0:
        df = df.loc[~bad_cost_mask].reset_index(drop=True)
        print(f"WARN: Dropped {bad_cost_count} rows with non-finite cost values before normalization")

    if len(df) == 0:
        raise ValueError("No rows left after removing non-finite cost values.")

    # Fit categorical encoders
    db_enc = LabelEncoder().fit(df["db_engine"])
    hw_enc = LabelEncoder().fit(df["hardware"])
    print(f"INFO: Fitted encoders: {len(db_enc.classes_)} DB engines, {len(hw_enc.classes_)} hardware configs")

    # Fit feature scaler on ALL domains (no target leakage since we only fit on features)
    fit_data = df[feat_cols_filtered].values
    print(f"INFO: Fitting RobustScaler on all {len(fit_data)} samples")
    feat_scaler = RobustScaler(quantile_range=(5.0, 95.0)).fit(fit_data)
    
    # Save the true raw cost prior to transform/normalization for MAPE/RMSE logging
    df["raw_cost"] = df[LABEL_COL].values

    # Optional target transform before normalization
    target_transform = {"type": "none"}
    if log_target:
        df[LABEL_COL] = np.log1p(np.maximum(df[LABEL_COL].values, 0.0))
        target_transform = {"type": "log1p"}

    # Cost normalization setup (finite-safe)
    global_mu = float(df[LABEL_COL].mean())
    global_sigma = float(df[LABEL_COL].std())
    if (not np.isfinite(global_mu)) or (not np.isfinite(global_sigma)) or (global_sigma < 1e-8):
        finite_cost = df[LABEL_COL].to_numpy()
        finite_cost = finite_cost[np.isfinite(finite_cost)]
        if finite_cost.size == 0:
            raise ValueError("Could not compute finite cost normalization stats.")
        global_mu = float(np.mean(finite_cost))
        global_sigma = float(np.std(finite_cost, ddof=1)) if finite_cost.size > 1 else 1.0
        if (not np.isfinite(global_sigma)) or (global_sigma < 1e-8):
            global_sigma = 1.0

    normalization_mode = normalization_mode.lower().strip()
    if normalization_mode == "global":
        cost_mu_map = {"__global__": global_mu}
        cost_sigma_map = {"__global__": global_sigma}
        print("\nINFO: Cost normalization mode: GLOBAL")
        print(f"   mu={global_mu:.4f}, sigma={global_sigma:.4f}")
    elif normalization_mode == "per_engine":
        stats_by_engine = df.groupby("db_engine")[LABEL_COL].agg(["mean", "std"])
        stats_by_engine["mean"] = stats_by_engine["mean"].replace([np.inf, -np.inf], np.nan).fillna(global_mu)
        stats_by_engine["std"] = stats_by_engine["std"].replace([np.inf, -np.inf], np.nan).fillna(global_sigma).clip(lower=1e-8)
        cost_mu_map = {k: float(v) for k, v in stats_by_engine["mean"].to_dict().items()}
        cost_sigma_map = {k: float(v) for k, v in stats_by_engine["std"].to_dict().items()}

        print("\nINFO: Cost normalization mode: PER_ENGINE")
        for engine in sorted(cost_mu_map.keys()):
            cnt = int((df["db_engine"] == engine).sum())
            print(f"   {engine}: n={cnt}, mu={cost_mu_map[engine]:.4f}, sigma={cost_sigma_map[engine]:.4f}")
    else:
        raise ValueError("normalization_mode must be 'global' or 'per_engine'")

    # Apply normalization
    if "__global__" in cost_mu_map:
        mu_series = pd.Series(cost_mu_map["__global__"], index=df.index)
        sigma_series = pd.Series(cost_sigma_map["__global__"], index=df.index).clip(lower=1e-8)
    else:
        mu_series = df["db_engine"].map(cost_mu_map).fillna(global_mu)
        sigma_series = df["db_engine"].map(cost_sigma_map).fillna(global_sigma).clip(lower=1e-8)
    df[LABEL_COL] = (df[LABEL_COL] - mu_series) / sigma_series

    # Final guard: remove any non-finite normalized targets
    bad_norm_mask = ~np.isfinite(df[LABEL_COL].to_numpy())
    bad_norm_count = int(bad_norm_mask.sum())
    if bad_norm_count > 0:
        df = df.loc[~bad_norm_mask].reset_index(drop=True)
        print(f"WARN: Dropped {bad_norm_count} rows with non-finite normalized costs")

    print(f"   Normalized overall: mu={df[LABEL_COL].mean():.4f}, sigma={df[LABEL_COL].std():.4f}")
    print(f"   Range: [{df[LABEL_COL].min():.4f}, {df[LABEL_COL].max():.4f}]")

    # Verification: normalized stats by engine family and domain groups
    mysql_mask = df[DOMAIN_COL].isin([0, 1])
    postgres_mask = df[DOMAIN_COL].isin([2, 3])
    if mysql_mask.any():
        print(
            f"   Normalized MySQL domains (0,1): mu={df.loc[mysql_mask, LABEL_COL].mean():.4f}, "
            f"sigma={df.loc[mysql_mask, LABEL_COL].std():.4f}"
        )
    if postgres_mask.any():
        print(
            f"   Normalized PostgreSQL domains (2,3): mu={df.loc[postgres_mask, LABEL_COL].mean():.4f}, "
            f"sigma={df.loc[postgres_mask, LABEL_COL].std():.4f}"
        )

    print("   Per-engine normalized mean/std:")
    print(df.groupby("db_engine")[LABEL_COL].agg(["mean", "std"]).round(4))

    return (
        df, feat_cols_filtered, mask_cols, qp_cols, db_enc, hw_enc, feat_scaler,
        cost_mu_map, cost_sigma_map, target_transform
    )

print("INFO: Preprocessing functions defined")

class CostModelDataset(Dataset):
    """
    PyTorch dataset for cost model training.
    
    IMPORTANT: Costs should be ALREADY NORMALIZED in preprocessing.
    This class just packages data for PyTorch - no transformation applied.
    """
    def __init__(self, df, feat_cols, mask_cols, qp_cols,
                 db_enc, hw_enc, feat_scaler,
                 labeled_target_idx=None, raw_costs=None):
        X_feat = feat_scaler.transform(df[feat_cols].values.astype(np.float32))
        X_mask = df[mask_cols].values.astype(np.float32)
        self.x_feat = X_feat
        self.x_mask = X_mask
        self.qp_emb = df[qp_cols].values.astype(np.float32)

        # Conditioning inputs for regression head
        self.db_oh = np.eye(len(db_enc.classes_), dtype=np.float32)[
                         db_enc.transform(df["db_engine"])]
        self.hw_oh = np.eye(len(hw_enc.classes_), dtype=np.float32)[
                         hw_enc.transform(df["hardware"])]
        self.ram = df["ram_gb"].values.astype(np.float32).reshape(-1, 1)

        # Labels and domain IDs (costs already normalized)
        self.cost = df[LABEL_COL].values.astype(np.float32)
        self.domain = df[DOMAIN_COL].values.astype(np.int64)
        
        # Raw costs for unnormalized metric evaluation during training
        if raw_costs is not None:
            self.raw_cost = raw_costs.astype(np.float32)
        else:
            self.raw_cost = self.cost

        # has_label: 1 = use this row in task loss
        self.has_label = (self.domain == SOURCE_DOMAIN).astype(np.float32)
        if labeled_target_idx is not None:
            self.has_label[list(labeled_target_idx)] = 1.0

    def __len__(self):
        return len(self.x_feat)

    def __getitem__(self, idx):
        return {
            "x_feat": torch.tensor(self.x_feat[idx]),
            "x_mask": torch.tensor(self.x_mask[idx]),
            "qp_emb": torch.tensor(self.qp_emb[idx]),
            "db_oh": torch.tensor(self.db_oh[idx]),
            "hw_oh": torch.tensor(self.hw_oh[idx]),
            "ram": torch.tensor(self.ram[idx]),
            "cost": torch.tensor(self.cost[idx]),
            "raw_cost": torch.tensor(self.raw_cost[idx]),
            "domain": torch.tensor(self.domain[idx]),
            "has_label": torch.tensor(self.has_label[idx]),
        }


def build_sampler(dataset):
    """
    WeightedRandomSampler for balanced domain batches.
    Prevents source domain (largest) from dominating every batch.
    """
    domains = dataset.domain
    class_counts = np.bincount(domains, minlength=4).astype(np.float32)
    weights = 1.0 / class_counts[domains]
    return WeightedRandomSampler(weights, num_samples=len(dataset), replacement=True)

print("INFO: Dataset and sampler defined")




## 4. Training Step (train_epoch)

```python
def train_epoch(model, loader, optimizer, device, lambda_coral=0.0, alpha_grl=0.0,
                dann_weight=0.1, alpha_target=0.0, domain_class_weights=None):
    """Train for one epoch with Huber + CORAL + DANN (GRL) losses.
    
    dann_weight: multiplier on the DANN BCE loss to keep it proportional
                 to the task loss (~0.03). Without this, BCE (~0.6) drowns
                 out the regression signal.
    """
    model.train()

    total_loss = task_sum = coral_sum = dann_sum = 0.0
    domain_correct = domain_total = 0
    n = 0

    huber = torch.nn.SmoothL1Loss(beta=1.0)

    for batch in loader:
        x_feat = batch["x_feat"].to(device)
        x_mask = batch["x_mask"].to(device)
        qp_emb = batch["qp_emb"].to(device)
        db_oh = batch["db_oh"].to(device)
        hw_oh = batch["hw_oh"].to(device)
        ram = batch["ram"].to(device)
        cost = batch["cost"].to(device)
        domain = batch["domain"].to(device)
        has_label = batch["has_label"].to(device)

        # Forward pass with GRL alpha
        cost_pred, z, domain_logits = model(
            x_feat, x_mask, qp_emb, db_oh, hw_oh, ram, alpha_grl=alpha_grl
        )
        cost_pred = torch.clamp(cost_pred, min=-10, max=10)

        # --- Task loss: Huber on all labeled samples ---
        src_mask = has_label.bool()
        task_loss = (
            huber(cost_pred[src_mask], cost[src_mask])
            if src_mask.sum() > 0
            else torch.tensor(0.0, device=device)
        )

        # Also use labeled target samples if available
        tgt_mask = (~(domain == SOURCE_DOMAIN)) & has_label.bool()
        if alpha_target > 0 and tgt_mask.sum() > 0:
            task_loss = task_loss + alpha_target * huber(cost_pred[tgt_mask], cost[tgt_mask])

        # --- CORAL alignment loss on z ---
        src_z_mask = (domain == SOURCE_DOMAIN)
        tgt_z_mask = (~(domain == SOURCE_DOMAIN))
        if lambda_coral > 0 and src_z_mask.sum() > 2 and tgt_z_mask.sum() > 2:
            coral_l = coral_loss(z[src_z_mask], z[tgt_z_mask])
        else:
            coral_l = torch.tensor(0.0, device=device)

        # --- DANN domain classifier loss (BCE: source=1, target=0) ---
        if alpha_grl > 0:
            domain_labels = (domain == SOURCE_DOMAIN).float()  # 1=source, 0=target
            # Class-weight: target samples are ~25% of data, so upweight them
            n_src = (domain_labels == 1).sum().clamp(min=1)
            n_tgt = (domain_labels == 0).sum().clamp(min=1)
            pos_weight = (n_tgt.float() / n_src.float()).clamp(0.1, 10.0)
            bce_w = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            dann_l = bce_w(domain_logits, domain_labels)
            # Track domain classifier accuracy
            with torch.no_grad():
                preds = (domain_logits > 0).float()
                domain_correct += (preds == domain_labels).sum().item()
                domain_total += len(domain_labels)
        else:
            dann_l = torch.tensor(0.0, device=device)

        loss = task_loss + lambda_coral * coral_l + dann_weight * dann_l

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        task_sum += task_loss.item()
        coral_sum += coral_l.item()
        dann_sum += dann_l.item()
        n += 1

    if n == 0:
        return {"loss": float("nan"), "task_loss": float("nan"),
                "coral_loss": float("nan"), "dann_loss": float("nan"),
                "domain_acc": float("nan")}

    d_acc = domain_correct / max(domain_total, 1)
    return {
        "loss": total_loss / n, "task_loss": task_sum / n,
        "coral_loss": coral_sum / n, "dann_loss": dann_sum / n,
        "domain_acc": d_acc,
    }
```

## 5. Evaluation Utilities

In [ ]:
if False:
    from data import (
        CostModelDataset,
        DOMAIN_COL,
        LABEL_COL,
        QP_EMB_COL,
        expand_qp_emb,
        get_log_scale_cols,
    )
elif False:
    from src.data import (
        CostModelDataset,
        DOMAIN_COL,
        LABEL_COL,
        QP_EMB_COL,
        expand_qp_emb,
        get_log_scale_cols,
    )


def _robust_mape(yt, yp, cap_pct=200.0):
    """Compute MAPE with robust threshold and capping to prevent near-zero denominator explosion."""
    # Use 1% of median as threshold to exclude near-zero costs
    median_abs = np.median(np.abs(yt))
    threshold = max(median_abs * 0.01, 0.1)  # At least 0.1 to be safe
    nz = np.abs(yt) > threshold
    if not nz.any():
        return float("nan")
    ape = np.abs((yp[nz] - yt[nz]) / yt[nz]) * 100.0
    # Cap individual APE at cap_pct to prevent outlier explosion
    ape = np.minimum(ape, cap_pct)
    return float(np.mean(ape))


def _per_domain_metrics(cost_true_orig, cost_pred_orig, domain):
    per_domain = {}
    for d in sorted(set(domain.tolist())):
        mask = domain == d
        if mask.sum() == 0:
            continue
        yt = cost_true_orig[mask]
        yp = cost_pred_orig[mask]
        rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
        mape = _robust_mape(yt, yp)
        rho, _ = spearmanr(yt, yp) if len(yt) > 2 else (float("nan"), 1.0)
        per_domain[str(int(d))] = {
            "n": int(mask.sum()),
            "rmse": rmse,
            "mape": mape,
            "spearman": float(rho),
        }
    return per_domain


def evaluate_on_raw_dataframe(
    model,
    raw_df,
    db_enc,
    hw_enc,
    feat_scaler,
    feat_cols,
    mask_cols,
    qp_cols,
    cost_mu_map,
    cost_sigma_map,
    batch_size=256,
    target_transform=None,
):
    eval_df = raw_df.copy()

    if eval_df[QP_EMB_COL].dtype == object and isinstance(eval_df[QP_EMB_COL].iloc[0], str):
        eval_df[QP_EMB_COL] = eval_df[QP_EMB_COL].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )

    log_scale_cols = get_log_scale_cols(eval_df)
    for col in log_scale_cols:
        eval_df[col] = np.log1p(eval_df[col].clip(lower=0))

    eval_df[feat_cols] = eval_df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eval_df[mask_cols] = eval_df[mask_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    for col in feat_cols:
        eval_df[col] = np.clip(eval_df[col], -1e4, 1e4)

    emb_df_eval, _ = expand_qp_emb(eval_df)
    emb_df_eval = emb_df_eval.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eval_df = pd.concat([eval_df.reset_index(drop=True), emb_df_eval.reset_index(drop=True)], axis=1)

    eval_df[LABEL_COL] = pd.to_numeric(eval_df[LABEL_COL], errors="coerce")
    finite_cost_mask = np.isfinite(eval_df[LABEL_COL].to_numpy())
    if not finite_cost_mask.all():
        eval_df = eval_df.loc[finite_cost_mask].reset_index(drop=True)

    if len(eval_df) == 0:
        return {
            "mse_orig": float("nan"),
            "rmse_orig": float("nan"),
            "mae_orig": float("nan"),
            "nrmse_pct": float("nan"),
            "mape_pct": float("nan"),
            "smape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_pval": float("nan"),
            "domain_acc": float("nan"),
            "n_samples": 0,
            "pooled": {"rmse": float("nan"), "mape": float("nan"), "spearman": float("nan"), "nrmse": float("nan")},
            "per_domain": {},
        }

    # CRITICAL: Apply the same target transform as preprocess() BEFORE normalizing
    # The mu/sigma were computed on log-transformed costs, so we must log-transform here too
    if target_transform and target_transform.get("type") == "log1p":
        eval_df[LABEL_COL] = np.log1p(np.maximum(eval_df[LABEL_COL].values, 0.0))

    if "__global__" in cost_mu_map:
        global_mu = float(cost_mu_map["__global__"])
        global_sigma = float(max(cost_sigma_map["__global__"], 1e-8))
        mu_eval = pd.Series(global_mu, index=eval_df.index)
        sigma_eval = pd.Series(global_sigma, index=eval_df.index)
    else:
        global_mu = float(np.mean(list(cost_mu_map.values())))
        global_sigma = float(np.mean(list(cost_sigma_map.values())))
        mu_eval = eval_df["db_engine"].map(cost_mu_map).fillna(global_mu)
        sigma_eval = eval_df["db_engine"].map(cost_sigma_map).fillna(global_sigma).clip(lower=1e-8)
    eval_df[LABEL_COL] = (eval_df[LABEL_COL] - mu_eval) / sigma_eval

    finite_norm_mask = np.isfinite(eval_df[LABEL_COL].to_numpy())
    if not finite_norm_mask.all():
        eval_df = eval_df.loc[finite_norm_mask].reset_index(drop=True)

    if len(eval_df) == 0:
        return {
            "mse_orig": float("nan"),
            "rmse_orig": float("nan"),
            "mae_orig": float("nan"),
            "nrmse_pct": float("nan"),
            "mape_pct": float("nan"),
            "smape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_pval": float("nan"),
            "domain_acc": float("nan"),
            "n_samples": 0,
            "pooled": {"rmse": float("nan"), "mape": float("nan"), "spearman": float("nan"), "nrmse": float("nan")},
            "per_domain": {},
        }

    device = next(model.parameters()).device
    eval_ds = CostModelDataset(eval_df, feat_cols, mask_cols, qp_cols, db_enc, hw_enc, feat_scaler)
    eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model.eval()
    all_cost_pred = []
    all_cost_true = []
    all_domain_pred = []
    all_domain_true = []

    with torch.no_grad():
        for batch in eval_loader:
            x_feat = batch["x_feat"].to(device)
            x_mask = batch["x_mask"].to(device)
            qp_emb = batch["qp_emb"].to(device)
            cost_pred, _, domain_logits = model(
                x_feat,
                x_mask,
                qp_emb,
                batch["db_oh"].to(device),
                batch["hw_oh"].to(device),
                batch["ram"].to(device),
                alpha_grl=0.0,
            )
            domain_pred = (domain_logits > 0).long()  # 1=source, 0=target
            all_cost_pred.append(cost_pred.cpu().numpy())
            all_cost_true.append(batch["cost"].numpy())
            all_domain_pred.append(domain_pred.cpu().numpy())
            all_domain_true.append(batch["domain"].numpy())

    cost_pred_norm = np.concatenate(all_cost_pred)
    cost_true_norm = np.concatenate(all_cost_true)
    domain_pred_all = np.concatenate(all_domain_pred)
    domain_true_all = np.concatenate(all_domain_true)

    if "__global__" in cost_mu_map:
        mu_eval_arr = np.full(len(eval_df), float(cost_mu_map["__global__"]), dtype=np.float32)
        sigma_eval_arr = np.full(
            len(eval_df), float(max(cost_sigma_map["__global__"], 1e-8)), dtype=np.float32
        )
    else:
        global_mu = float(np.mean(list(cost_mu_map.values())))
        global_sigma = float(np.mean(list(cost_sigma_map.values())))
        mu_eval_arr = eval_df["db_engine"].map(cost_mu_map).fillna(global_mu).to_numpy(dtype=np.float32)
        sigma_eval_arr = (
            eval_df["db_engine"].map(cost_sigma_map).fillna(global_sigma).clip(lower=1e-8).to_numpy(dtype=np.float32)
        )

    cost_pred_orig = cost_pred_norm * sigma_eval_arr + mu_eval_arr
    cost_true_orig = cost_true_norm * sigma_eval_arr + mu_eval_arr

    if target_transform and target_transform.get("type") == "log1p":
        cost_pred_orig = np.expm1(np.clip(cost_pred_orig, a_min=None, a_max=20.0))
        cost_true_orig = np.expm1(np.clip(cost_true_orig, a_min=None, a_max=20.0))

    finite_pair_mask = np.isfinite(cost_pred_orig) & np.isfinite(cost_true_orig)
    if not finite_pair_mask.all():
        cost_pred_orig = cost_pred_orig[finite_pair_mask]
        cost_true_orig = cost_true_orig[finite_pair_mask]
        domain_pred_all = domain_pred_all[finite_pair_mask]
        domain_true_all = domain_true_all[finite_pair_mask]

    if len(cost_true_orig) == 0:
        return {
            "mse_orig": float("nan"),
            "rmse_orig": float("nan"),
            "mae_orig": float("nan"),
            "nrmse_pct": float("nan"),
            "mape_pct": float("nan"),
            "smape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_pval": float("nan"),
            "domain_acc": float("nan"),
            "n_samples": 0,
            "pooled": {"rmse": float("nan"), "mape": float("nan"), "spearman": float("nan"), "nrmse": float("nan")},
            "per_domain": {},
        }

    err = cost_pred_orig - cost_true_orig
    abs_err = np.abs(err)

    mse_orig = float(np.mean(err ** 2))
    rmse_orig = float(np.sqrt(mse_orig))
    mae_orig = float(np.mean(abs_err))

    orig_range = float(np.ptp(cost_true_orig))
    nrmse_pct = float((rmse_orig / (orig_range + 1e-8)) * 100.0)

    mape_pct = _robust_mape(cost_true_orig, cost_pred_orig)

    smape_pct = float(
        np.mean((2.0 * abs_err) / (np.abs(cost_true_orig) + np.abs(cost_pred_orig) + 1e-8)) * 100.0
    )

    spearman_rho, spearman_pval = spearmanr(cost_true_orig, cost_pred_orig)
    domain_acc = float(np.mean(domain_pred_all == domain_true_all))

    per_domain = _per_domain_metrics(cost_true_orig, cost_pred_orig, domain_true_all)

    return {
        "mse_orig": mse_orig,
        "rmse_orig": rmse_orig,
        "mae_orig": mae_orig,
        "nrmse_pct": nrmse_pct,
        "mape_pct": mape_pct,
        "smape_pct": smape_pct,
        "spearman_rho": float(spearman_rho),
        "spearman_pval": float(spearman_pval),
        "domain_acc": domain_acc,
        "n_samples": int(len(cost_true_orig)),
        "pooled": {
            "rmse": rmse_orig,
            "mape": mape_pct,
            "spearman": float(spearman_rho),
            "nrmse": nrmse_pct,
        },
        "per_domain": per_domain,
    }


print("INFO: evaluate_on_raw_dataframe ready")



## 6. CORAL + DANN Model Components

In [ ]:
def coral_loss(source_features, target_features):
    """
    CORAL loss (Sun & Saenko, ECCV 2016).
    Aligns second-order statistics (covariance matrices) of source and target.
    Much more numerically stable than RSD/adversarial approaches for regression.
    """
    d = source_features.size(1)
    ns = source_features.size(0)
    nt = target_features.size(0)

    # Center
    src = source_features - source_features.mean(0, keepdim=True)
    tgt = target_features - target_features.mean(0, keepdim=True)

    # Covariance matrices
    cs = (src.T @ src) / max(ns - 1, 1)
    ct = (tgt.T @ tgt) / max(nt - 1, 1)

    # Frobenius norm of difference, normalized by 4*d^2
    loss = torch.sum((cs - ct) ** 2) / (4 * d * d)
    return loss

print("INFO: CORAL loss defined")


# ======================== Gradient Reversal Layer ========================
class GradientReversalFunction(Function):
    """Gradient Reversal Layer (Ganin et al., JMLR 2016).
    Forward: identity.  Backward: negate and scale by alpha."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    """Wraps GradientReversalFunction for use in nn.Sequential / forward()."""
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

    def set_alpha(self, alpha):
        self.alpha = alpha

print("INFO: Gradient Reversal Layer defined")


# ======================== SAINT Encoder ========================
class SAINTBlock(nn.Module):
    """Single SAINT transformer block with feature attention only.
    Row attention removed for stability and speed on small datasets."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.feat_attn = nn.MultiheadAttention(d_model, n_heads,
                                                dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Feature attention (within sample, across features)
        res = x
        x2, _ = self.feat_attn(x, x, x)
        x = self.ln1(res + x2)

        # Feed-forward
        res = x
        x = self.ln3(res + self.ff(x))
        return x


class SAINTEncoder(nn.Module):
    """Gf: tabular features -> latent vector z"""
    def __init__(self, num_features, d_model=128, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.num_features = num_features
        self.feat_embed = nn.Embedding(num_features, d_model)
        self.val_proj = nn.Linear(1, d_model)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout)
                                     for _ in range(n_layers)])
        self.ln_out = nn.LayerNorm(d_model)

    def forward(self, x):
        B, F = x.shape
        feat_ids = torch.arange(F, device=x.device).unsqueeze(0).expand(B, -1)
        tokens = self.feat_embed(feat_ids) + self.val_proj(x.unsqueeze(-1))
        for block in self.blocks:
            tokens = block(tokens)
        z = self.ln_out(tokens).mean(dim=1)
        return z

print("INFO: SAINT encoder defined")


# ======================== Regression Head (V4 — simpler, proven) ========================
class RegressionHead(nn.Module):
    """Gy: [z | plan_emb | masks | db_engine_oh | hardware_oh | ram_gb] -> predicted cost
    
    V4 architecture: 256->128->64->1 (simpler, better performance than V5's wider head).
    """
    def __init__(self, z_dim, plan_emb_dim, n_masks, n_db_engines, n_hardware, dropout=0.1, n_domains=4):
        super().__init__()
        cond_dim = z_dim + plan_emb_dim + n_masks + n_db_engines + n_hardware + 1
        self.net = nn.Sequential(
            nn.Linear(cond_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, z, plan_emb, masks, db_engine_oh, hardware_oh, ram_gb, domain_ids=None):
        x = torch.cat([z, plan_emb, masks, db_engine_oh, hardware_oh, ram_gb], dim=1)
        return self.net(x).squeeze(1)


print("INFO: Regression head defined")


# ======================== Domain Classifier ========================
class DomainClassifier(nn.Module):
    """Gd: z -> domain prediction (binary: source vs target).
    
    Small MLP to classify whether features come from source or target domain.
    Connected via GRL so the encoder learns to confuse it.
    """
    def __init__(self, z_dim, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.grl = GradientReversalLayer(alpha=1.0)
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),  # Binary: source vs target
        )

    def forward(self, z, alpha=1.0):
        self.grl.set_alpha(alpha)
        z_rev = self.grl(z)
        return self.net(z_rev).squeeze(1)

print("INFO: Domain classifier defined")


# ======================== DANN Cost Model ========================
class DANNCostModel(nn.Module):
    """Full DANN: SAINT encoder + regression head + domain classifier (GRL).
    
    Three-branch architecture:
    - Gf (SAINT encoder): shared feature extractor -> z
    - Gy (regression head): z -> cost prediction  
    - Gd (domain classifier via GRL): z -> source/target prediction
    
    CORAL alignment is computed externally on z during training.
    """
    def __init__(self, num_features, n_masks, n_db_engines, n_hardware,
                 d_model=128, n_heads=4, n_layers=2, n_domains=4, dropout=0.1):
        super().__init__()
        self.Gf = SAINTEncoder(num_features, d_model, n_heads, n_layers, dropout)
        self.plan_proj = nn.Sequential(
            nn.Linear(384, d_model // 2),
            nn.LayerNorm(d_model // 2),
            nn.GELU()
        )
        self.Gy = RegressionHead(d_model, d_model // 2, n_masks, n_db_engines, n_hardware, dropout, n_domains)
        self.Gd = DomainClassifier(d_model, hidden_dim=128, dropout=dropout)

    def forward(self, x_feat, x_mask, qp_emb, db_engine_oh, hardware_oh, ram_gb, 
                domain_ids=None, alpha_grl=0.0):
        """Forward pass returning cost prediction, latent z, and domain logits.
        
        Args:
            alpha_grl: GRL scaling factor. 0 = no adversarial training.
        """
        z = self.Gf(x_feat)
        p = self.plan_proj(qp_emb)
        cost_pred = self.Gy(z, p, x_mask, db_engine_oh, hardware_oh, ram_gb, domain_ids)
        domain_logits = self.Gd(z, alpha=alpha_grl)
        return cost_pred, z, domain_logits

    @torch.no_grad()
    def predict(self, x_feat, x_mask, qp_emb, db_engine_oh, hardware_oh, ram_gb, domain_ids=None):
        """Inference (no domain classifier needed)"""
        self.eval()
        z = self.Gf(x_feat)
        p = self.plan_proj(qp_emb)
        return self.Gy(z, p, x_mask, db_engine_oh, hardware_oh, ram_gb, domain_ids)

print("INFO: DANNCostModel defined")





## 7. Training Pipeline

In [ ]:
if False:
    from data import (
        CostModelDataset,
        DOMAIN_COL,
        SOURCE_DOMAIN,
        build_sampler,
        preprocess,
    )
    from eval import evaluate_on_raw_dataframe
    from models import DANNCostModel, coral_loss
elif False:
    from src.data import (
        CostModelDataset,
        DOMAIN_COL,
        SOURCE_DOMAIN,
        build_sampler,
        preprocess,
    )
    from src.eval import evaluate_on_raw_dataframe
    from src.models import DANNCostModel, coral_loss


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def compute_lambda_coral(p, max_lambda=0.05):
    """
    Lambda schedule for CORAL alignment.
    Conservative: warm up slowly and cap at a low max to avoid harming regression.
    """
    if p < 0.2:
        return 0.0  # Pure supervised warmup first
    elif p < 0.5:
        return ((p - 0.2) / 0.3) * max_lambda
    return max_lambda


def compute_lambda_dann(p, alpha=10.0, max_dann=0.3):
    """
    DANN lambda schedule (adapted from Ganin et al., JMLR 2016).
    Smoothly increases from 0 to max_dann.
    Delayed start: first 20% of training is pure supervised (GRL off).
    Capped at max_dann to prevent adversarial loss from overwhelming task loss.
    """
    import math
    if p < 0.2:
        return 0.0  # Pure supervised warmup
    p_adj = (p - 0.2) / 0.8  # Rescale remaining to [0, 1]
    raw = 2.0 / (1.0 + math.exp(-alpha * p_adj)) - 1.0
    return raw * max_dann


print("INFO: compute_lambda defined")


def _domain_count_map(df, col=DOMAIN_COL):
    if col not in df.columns or len(df) == 0:
        return {}
    vc = df[col].value_counts().sort_index()
    return {int(k): int(v) for k, v in vc.items()}


def _format_domain_counts(counts):
    return " | ".join(f"D{d}={n}" for d, n in sorted(counts.items()))


print("INFO: Training helpers defined")

def evaluate(model, loader, device):
    """Evaluate on labeled samples (for validation)."""
    model.eval()
    all_cp, all_ct, all_dt, all_hl, all_dl, all_dom = [], [], [], [], [], []

    with torch.no_grad():
        for batch in loader:
            x_feat = batch["x_feat"].to(device)
            x_mask = batch["x_mask"].to(device)
            qp_emb = batch["qp_emb"].to(device)
            domain = batch["domain"].to(device)
            cp, _, dl = model(
                x_feat, x_mask, qp_emb,
                batch["db_oh"].to(device),
                batch["hw_oh"].to(device),
                batch["ram"].to(device),
                alpha_grl=0.0,  # No GRL during eval
            )
            cp = torch.clamp(cp, min=-10, max=10)

            all_cp.append(cp.cpu())
            all_ct.append(batch["cost"])
            all_dt.append(batch["domain"])
            all_hl.append(batch["has_label"])
            all_dl.append(dl.cpu())
            all_dom.append(domain.cpu())

    cp = torch.cat(all_cp)
    ct = torch.cat(all_ct)
    dt = torch.cat(all_dt)
    hl = torch.cat(all_hl).bool()
    dl = torch.cat(all_dl)
    dom = torch.cat(all_dom)

    results = {}
    for d in range(4):
        mask = (dt == d) & hl
        if mask.sum() > 0:
            mse_val = F.mse_loss(cp[mask], ct[mask]).item()
            results[f"mse_domain_{d}"] = mse_val

    # Domain classifier accuracy (how well can we distinguish domains?)
    domain_labels = (dom == SOURCE_DOMAIN).float()
    domain_preds = (dl > 0).float()
    d_acc = (domain_preds == domain_labels).float().mean().item()
    results["domain_acc"] = d_acc
    results["h_divergence_approx"] = max(0.0, 2 * (2 * d_acc - 1))
    return results


def evaluate_all(model, loader, device, cost_mu_map=None, cost_sigma_map=None, db_classes=None, target_transform=None):
    """Evaluate on all samples (including unlabeled target)."""
    model.eval()
    all_cp, all_ct, all_dt, all_rt, all_db = [], [], [], [], []

    with torch.no_grad():
        for batch in loader:
            x_feat = batch["x_feat"].to(device)
            x_mask = batch["x_mask"].to(device)
            qp_emb = batch["qp_emb"].to(device)
            cp, _, _ = model(
                x_feat, x_mask, qp_emb,
                batch["db_oh"].to(device),
                batch["hw_oh"].to(device),
                batch["ram"].to(device),
                alpha_grl=0.0,
            )

            cp = torch.clamp(cp, min=-10, max=10)

            all_cp.append(cp.cpu().numpy())
            all_ct.append(batch["cost"].numpy())
            all_dt.append(batch["domain"].numpy())
            if "raw_cost" in batch:
                all_rt.append(batch["raw_cost"].numpy())
            all_db.append(batch["db_oh"].argmax(1).numpy())

    cp = np.concatenate(all_cp)
    ct = np.concatenate(all_ct)
    dt = np.concatenate(all_dt)

    if all_rt:
        rt = np.concatenate(all_rt)
        db = np.concatenate(all_db)
        
        cp_orig = np.zeros_like(cp)
        if cost_mu_map and "__global__" in cost_mu_map:
            cp_orig = cp * cost_sigma_map["__global__"] + cost_mu_map["__global__"]
        elif cost_mu_map and db_classes is not None:
            for idx, c in enumerate(db_classes):
                mask = db == idx
                sigma = cost_sigma_map.get(c, 1.0)
                mu = cost_mu_map.get(c, 0.0)
                cp_orig[mask] = cp[mask] * sigma + mu
        else:
            cp_orig = cp.copy()
            
        if target_transform and target_transform.get("type") == "log1p":
            cp_orig = np.expm1(np.clip(cp_orig, a_min=None, a_max=20.0))
            
        cp_orig = np.clip(cp_orig, 0.0, 1e12)

    results = {}
    for d in range(4):
        mask = dt == d
        if mask.sum() > 0:
            mse_val = float(np.mean((cp[mask] - ct[mask])**2))
            results[f"mse_domain_{d}_all"] = mse_val
            
            if all_rt:
                rt_d = rt[mask]
                cp_d = cp_orig[mask]
                rmse = float(np.sqrt(np.mean((rt_d - cp_d)**2)))
                # Robust MAPE: threshold at 1% of median, cap at 200%
                median_abs = np.median(np.abs(rt_d))
                threshold = max(median_abs * 0.01, 0.1)
                nz = np.abs(rt_d) > threshold
                if nz.any():
                    ape = np.abs((cp_d[nz] - rt_d[nz]) / rt_d[nz]) * 100.0
                    ape = np.minimum(ape, 200.0)
                    mape = float(np.mean(ape))
                else:
                    mape = float("nan")
                results[f"rmse_domain_{d}_all"] = rmse
                results[f"mape_domain_{d}_all"] = mape
                # Spearman rank correlation (key metric for ranking-based labeling)
                if len(rt_d) > 2:
                    rho, _ = spearmanr(rt_d, cp_d)
                    results[f"spearman_domain_{d}_all"] = float(rho) if np.isfinite(rho) else float("nan")
                else:
                    results[f"spearman_domain_{d}_all"] = float("nan")

    return results


print("INFO: Training helpers defined")


def train(
    da,
    n_epochs=100,
    batch_size=256,
    lr=5e-4,
    d_model=128,
    n_heads=4,
    n_layers=2,
    dropout=0.15,
    alpha_target=1.0,
    labeled_target_frac=1.0,
    normalization_mode="per_engine",
    log_target=True,
    lambda_coral_max=0.05,
    use_dann=True,
    use_sampler=True,
    seed=42,
    device_str="cpu",
):
    """
    Train the DANN cost model with CORAL + adversarial GRL alignment.

    Architecture (V6 - True DANN):
    1. SAINT encoder -> shared features z
    2. Regression head (Gy): z -> cost prediction (Huber loss)
    3. Domain classifier (Gd) via GRL: z -> source/target (BCE loss)
    4. CORAL: aligns covariance matrices of source/target z
    
    The GRL forces the encoder to learn domain-invariant features.
    CORAL provides complementary second-order statistical alignment.
    """
    set_seed(seed)
    device = torch.device(device_str if torch.cuda.is_available() or device_str == "cpu" else "cpu")

    (
        df,
        feat_cols,
        mask_cols,
        qp_cols,
        db_enc,
        hw_enc,
        feat_scaler,
        cost_mu_map,
        cost_sigma_map,
        target_transform,
    ) = preprocess(da, normalization_mode=normalization_mode, log_target=log_target)

    train_idx, val_idx = train_test_split(
        np.arange(len(df)),
        test_size=0.15,
        stratify=df[DOMAIN_COL].values,
        random_state=seed,
    )

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    split_sizes = {"train": len(train_df), "val": len(val_df)}
    split_domain_counts = {
        "train": _domain_count_map(train_df),
        "val": _domain_count_map(val_df),
    }

    # Semi-supervised: label a fraction of target domain samples
    labeled_target_idx = None
    if labeled_target_frac > 0:
        rng = np.random.RandomState(seed)
        tgt_idx = np.where(train_df[DOMAIN_COL].values != SOURCE_DOMAIN)[0]
        if len(tgt_idx) > 0:
            n_label = max(1, int(len(tgt_idx) * labeled_target_frac))
            labeled_target_idx = rng.choice(tgt_idx, size=n_label, replace=False).tolist()
            print(f"INFO: Semi-supervised mode: labeling {n_label}/{len(tgt_idx)} target samples ({labeled_target_frac*100:.0f}%)")

    ds_kw = dict(
        feat_cols=feat_cols,
        mask_cols=mask_cols,
        qp_cols=qp_cols,
        db_enc=db_enc,
        hw_enc=hw_enc,
        feat_scaler=feat_scaler,
    )

    train_ds = CostModelDataset(train_df, **ds_kw, labeled_target_idx=labeled_target_idx, 
                                raw_costs=train_df["raw_cost"].values if "raw_cost" in train_df else None)
    val_ds = CostModelDataset(val_df, **ds_kw, 
                              raw_costs=val_df["raw_cost"].values if "raw_cost" in val_df else None)

    class_counts = np.bincount(train_ds.domain, minlength=4).astype(np.float32)
    class_counts = np.maximum(class_counts, 1.0)
    domain_class_weights = (len(train_ds) / (4.0 * class_counts)).astype(np.float32)
    domain_class_weights = domain_class_weights / domain_class_weights.mean()
    domain_class_weights = torch.tensor(domain_class_weights, dtype=torch.float32, device=device)

    if use_sampler:
        sampler = build_sampler(train_ds)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    num_feat_total = len(feat_cols)
    model = DANNCostModel(
        num_features=num_feat_total,
        n_masks=len(mask_cols),
        n_db_engines=len(db_enc.classes_),
        n_hardware=len(hw_enc.classes_),
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        n_domains=4,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

    best_mse = float("inf")
    best_spearman = -float("inf")
    best_state = None
    best_epoch = 0
    patience = 40
    patience_counter = 0
    total_steps = n_epochs * max(len(train_loader), 1)
    step = 0
    history = []
    start_time = time.time()

    for epoch in range(1, n_epochs + 1):
        p = step / max(total_steps, 1)
        lam_coral = compute_lambda_coral(p, max_lambda=lambda_coral_max)
        lam_dann = compute_lambda_dann(p, alpha=10.0) if use_dann else 0.0
        tr = train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            lambda_coral=lam_coral,
            alpha_grl=lam_dann,
            alpha_target=alpha_target,
            domain_class_weights=domain_class_weights,
        )
        step += len(train_loader)
        scheduler.step()

        val = evaluate(model, val_loader, device)
        val_all = evaluate_all(model, val_loader, device, 
                               cost_mu_map=cost_mu_map, 
                               cost_sigma_map=cost_sigma_map, 
                               db_classes=db_enc.classes_, 
                               target_transform=target_transform)

        src_mse = val.get("mse_domain_3", float("nan"))
        tgt_mse = np.nanmean(
            [val_all.get(f"mse_domain_{d}_all", float("nan")) for d in [0, 1, 2]]
        )
        
        src_rmse = val_all.get("rmse_domain_3_all", float("nan"))
        src_mape = val_all.get("mape_domain_3_all", float("nan"))
        tgt_rmse = np.nanmean([val_all.get(f"rmse_domain_{d}_all", float("nan")) for d in [0, 1, 2]])
        tgt_mape = np.nanmean([val_all.get(f"mape_domain_{d}_all", float("nan")) for d in [0, 1, 2]])
        
        # Macro validation MSE (for logging)
        all_val_mses = [val_all.get(f"mse_domain_{d}_all", float("nan")) for d in [0, 1, 2, 3]]
        macro_val_mse = np.nanmean(all_val_mses)

        # Macro Spearman (PRIMARY early stopping metric for ranking-based labeling)
        all_val_spearmans = [val_all.get(f"spearman_domain_{d}_all", float("nan")) for d in [0, 1, 2, 3]]
        macro_val_spearman = np.nanmean([s for s in all_val_spearmans if np.isfinite(s)]) if any(np.isfinite(s) for s in all_val_spearmans) else float("nan")

        # Print every epoch to monitor convergence
        if epoch % 5 == 0 or epoch == 1:
            d_acc_str = f"{tr.get('domain_acc', 0):.1%}" if use_dann else "off"
            spear_str = f"{macro_val_spearman:.4f}" if np.isfinite(macro_val_spearman) else "N/A"
            print(f"Epoch {epoch:03d}/{n_epochs} | "
                  f"TrLoss: {tr['loss']:.4f} (task={tr['task_loss']:.4f} "
                  f"coral={tr['coral_loss']:.4f} dann={tr['dann_loss']:.4f}) | "
                  f"Spear: {spear_str} | MSE: {macro_val_mse:.4f} | "
                  f"SrcMAPE: {src_mape:.1f}% | TgtMAPE: {tgt_mape:.1f}% | "
                  f"DomAcc: {d_acc_str} | "
                  f"lam_c: {lam_coral:.4f} lam_d: {lam_dann:.2f}")

        history.append(
            dict(
                epoch=epoch,
                lambda_coral=lam_coral,
                lambda_dann=lam_dann,
                train_loss=tr["loss"],
                task_loss=tr["task_loss"],
                coral_loss=tr["coral_loss"],
                dann_loss=tr["dann_loss"],
                train_domain_acc=tr.get("domain_acc", float("nan")),
                src_mse=src_mse,
                tgt_mse=tgt_mse,
                macro_val_mse=macro_val_mse,
                macro_val_spearman=macro_val_spearman,
                src_mape=src_mape,
                tgt_mape=tgt_mape,
                domain_acc=val["domain_acc"],
                h_div=val["h_divergence_approx"],
            )
        )

        # Early stopping: maximize Spearman (ranking accuracy) not minimize MSE
        if np.isfinite(macro_val_spearman) and macro_val_spearman > best_spearman:
            best_spearman = macro_val_spearman
            best_mse = macro_val_mse  # Track MSE too for logging
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience and epoch > 30:
            print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs). Best epoch: {best_epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    train_minutes = (time.time() - start_time) / 60.0
    history_df = pd.DataFrame(history)

    return {
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_mse": best_mse,
        "train_minutes": train_minutes,
        "split_sizes": split_sizes,
        "split_domain_counts": split_domain_counts,
        "feat_cols": feat_cols,
        "mask_cols": mask_cols,
        "qp_cols": qp_cols,
        "db_enc": db_enc,
        "hw_enc": hw_enc,
        "feat_scaler": feat_scaler,
        "cost_mu_map": cost_mu_map,
        "cost_sigma_map": cost_sigma_map,
        "target_transform": target_transform,
    }


print("INFO: Main training function defined")


def train_dann_custom(
    da,
    n_epochs=100,
    batch_size=256,
    lr=5e-4,
    alpha_target=1.0,
    labeled_target_frac=1.0,
    normalization_mode="per_engine",
    log_target=True,
    seed=42,
    device_str="cpu",
    lambda_coral_max=0.05,
    use_dann=True,
    use_sampler=True,
):
    return train(
        da,
        n_epochs=n_epochs,
        batch_size=batch_size,
        lr=lr,
        alpha_target=alpha_target,
        labeled_target_frac=labeled_target_frac,
        normalization_mode=normalization_mode,
        log_target=log_target,
        lambda_coral_max=lambda_coral_max,
        use_dann=use_dann,
        use_sampler=use_sampler,
        seed=seed,
        device_str=device_str,
    )


def train_and_eval(
    da,
    normalization_mode="per_engine",
    n_epochs=100,
    batch_size=256,
    lr=5e-4,
    seed=42,
    device_str="cpu",
    log_target=True,
    use_sampler=True,
    n_folds=5,
    labeled_target_frac=1.0,
    use_dann=True,
):
    """
    Complete training + evaluation pipeline with k-fold cross-validation.
    
    For each fold:
      - 1 fold = test set
      - remaining folds = train+val (with internal 85/15 split for early stopping)
    
    Returns aggregated metrics (mean ± std across folds), combined history, best model.
    """
    from sklearn.model_selection import StratifiedKFold

    print(f"INFO: Starting {n_folds}-fold CV with normalization_mode={normalization_mode}, "
          f"log_target={log_target}, use_sampler={use_sampler}, n_epochs={n_epochs}, lr={lr}")
    print(f"INFO: Total samples: {len(da)}")

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    
    fold_metrics = []
    fold_histories = []
    best_fold_mape = float("inf")
    best_model = None
    best_fold_idx = -1
    total_train_minutes = 0.0

    for fold_idx, (train_val_idx, test_idx) in enumerate(skf.split(da, da[DOMAIN_COL])):
        fold_num = fold_idx + 1
        print(f"\n{'='*60}")
        print(f"FOLD {fold_num}/{n_folds}")
        print(f"{'='*60}")

        da_train_val = da.iloc[train_val_idx].reset_index(drop=True)
        da_test = da.iloc[test_idx].reset_index(drop=True)
        print(f"Train+Val: {len(da_train_val)}, Test: {len(da_test)}")

        # Use a different seed for each fold to get diverse models
        fold_seed = seed + fold_idx

        out = train_dann_custom(
            da_train_val,
            n_epochs=n_epochs,
            batch_size=batch_size,
            lr=lr,
            labeled_target_frac=labeled_target_frac,
            use_dann=use_dann,
            normalization_mode=normalization_mode,
            log_target=log_target,
            use_sampler=use_sampler,
            seed=fold_seed,
            device_str=device_str,
        )

        model = out["model"]
        total_train_minutes += out["train_minutes"]

        print(f"Fold {fold_num} best epoch: {out['best_epoch']} | "
              f"macro-val MSE: {out['best_mse']:.6f} | "
              f"trained in {out['train_minutes']:.1f} min")

        # Evaluate on held-out test fold
        test_metrics = evaluate_on_raw_dataframe(
            model=model,
            raw_df=da_test,
            db_enc=out["db_enc"],
            hw_enc=out["hw_enc"],
            feat_scaler=out["feat_scaler"],
            feat_cols=out["feat_cols"],
            mask_cols=out["mask_cols"],
            qp_cols=out["qp_cols"],
            cost_mu_map=out["cost_mu_map"],
            cost_sigma_map=out["cost_sigma_map"],
            target_transform=out["target_transform"],
        )

        fold_mape = test_metrics.get("mape_pct", float("nan"))
        fold_spearman = test_metrics.get("spearman_rho", float("nan"))

        print(f"Fold {fold_num} TEST: MAPE={fold_mape:.2f}%, "
              f"RMSE={test_metrics.get('rmse_orig', float('nan')):.4f}, "
              f"Spearman={fold_spearman:.4f}")

        if "per_domain" in test_metrics:
            for d, dm in sorted(test_metrics["per_domain"].items()):
                print(f"  Domain {d}: MAPE={dm.get('mape', float('nan')):.2f}%, "
                      f"RMSE={dm.get('rmse', float('nan')):.4f}, "
                      f"Spearman={dm.get('spearman', float('nan')):.4f}, "
                      f"n={dm.get('n', 0)}")

        fold_metrics.append({
            "fold": fold_num,
            "pooled_mape": fold_mape,
            "pooled_rmse": test_metrics.get("rmse_orig", float("nan")),
            "pooled_spearman": fold_spearman,
            "best_epoch": out["best_epoch"],
            "train_minutes": out["train_minutes"],
            "per_domain": test_metrics.get("per_domain", {}),
            "pooled": test_metrics.get("pooled", {}),
        })

        # Add fold column to history
        hist = out["history_df"].copy()
        hist["fold"] = fold_num
        fold_histories.append(hist)

        # Track best fold for model selection
        if np.isfinite(fold_mape) and fold_mape < best_fold_mape:
            best_fold_mape = fold_mape
            best_model = model
            best_fold_idx = fold_num

    # === Aggregate results across folds ===
    print(f"\n{'='*60}")
    print(f"CROSS-VALIDATION RESULTS ({n_folds}-fold)")
    print(f"{'='*60}")

    pooled_mapes = [m["pooled_mape"] for m in fold_metrics if np.isfinite(m["pooled_mape"])]
    pooled_rmses = [m["pooled_rmse"] for m in fold_metrics if np.isfinite(m["pooled_rmse"])]
    pooled_spearmans = [m["pooled_spearman"] for m in fold_metrics if np.isfinite(m["pooled_spearman"])]

    print(f"\nPooled MAPE:     {np.mean(pooled_mapes):.2f}% ± {np.std(pooled_mapes):.2f}%")
    print(f"Pooled RMSE:     {np.mean(pooled_rmses):.4f} ± {np.std(pooled_rmses):.4f}")
    print(f"Pooled Spearman: {np.mean(pooled_spearmans):.4f} ± {np.std(pooled_spearmans):.4f}")

    # Per-domain aggregation
    all_domains = set()
    for m in fold_metrics:
        all_domains.update(m["per_domain"].keys())

    print(f"\nPer-domain results (mean ± std across {n_folds} folds):")
    domain_summary = {}
    for d in sorted(all_domains):
        d_mapes = [m["per_domain"][d]["mape"] for m in fold_metrics if d in m["per_domain"] and np.isfinite(m["per_domain"][d].get("mape", float("nan")))]
        d_rmses = [m["per_domain"][d]["rmse"] for m in fold_metrics if d in m["per_domain"] and np.isfinite(m["per_domain"][d].get("rmse", float("nan")))]
        d_spearmans = [m["per_domain"][d]["spearman"] for m in fold_metrics if d in m["per_domain"] and np.isfinite(m["per_domain"][d].get("spearman", float("nan")))]
        d_ns = [m["per_domain"][d]["n"] for m in fold_metrics if d in m["per_domain"]]

        mape_str = f"{np.mean(d_mapes):.2f}% ± {np.std(d_mapes):.2f}%" if d_mapes else "N/A"
        rmse_str = f"{np.mean(d_rmses):.4f} ± {np.std(d_rmses):.4f}" if d_rmses else "N/A"
        spear_str = f"{np.mean(d_spearmans):.4f} ± {np.std(d_spearmans):.4f}" if d_spearmans else "N/A"
        n_str = f"{np.mean(d_ns):.0f}" if d_ns else "N/A"

        print(f"  Domain {d}: MAPE={mape_str} | RMSE={rmse_str} | Spearman={spear_str} | n≈{n_str}")

        domain_summary[d] = {
            "mape_mean": float(np.mean(d_mapes)) if d_mapes else float("nan"),
            "mape_std": float(np.std(d_mapes)) if d_mapes else float("nan"),
            "rmse_mean": float(np.mean(d_rmses)) if d_rmses else float("nan"),
            "rmse_std": float(np.std(d_rmses)) if d_rmses else float("nan"),
            "spearman_mean": float(np.mean(d_spearmans)) if d_spearmans else float("nan"),
            "spearman_std": float(np.std(d_spearmans)) if d_spearmans else float("nan"),
        }

    print(f"\nTotal training time: {total_train_minutes:.1f} minutes ({total_train_minutes/60:.1f} hours)")
    print(f"Best fold: {best_fold_idx} (MAPE={best_fold_mape:.2f}%)")

    # Combine histories
    history_df = pd.concat(fold_histories, ignore_index=True)

    return (
        {
            "pooled": {
                "mape": float(np.mean(pooled_mapes)) if pooled_mapes else float("nan"),
                "mape_std": float(np.std(pooled_mapes)) if pooled_mapes else float("nan"),
                "rmse": float(np.mean(pooled_rmses)) if pooled_rmses else float("nan"),
                "rmse_std": float(np.std(pooled_rmses)) if pooled_rmses else float("nan"),
                "spearman": float(np.mean(pooled_spearmans)) if pooled_spearmans else float("nan"),
                "spearman_std": float(np.std(pooled_spearmans)) if pooled_spearmans else float("nan"),
            },
            "per_domain": domain_summary,
            "fold_metrics": fold_metrics,
            "domain_acc": float("nan"),
            "best_epoch": fold_metrics[best_fold_idx - 1]["best_epoch"] if best_fold_idx > 0 else 0,
            "train_minutes": total_train_minutes,
            "n_folds": n_folds,
        },
        history_df,
        best_model,
        domain_summary,
    )


print("INFO: run_ablation_suite ready")



## 8. Experiment Run

In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

try:
    df = pd.read_csv('/kaggle/input/olap-dataset-dnn-03/final_combined_olap.csv')
except FileNotFoundError:
    df = pd.read_csv('/kaggle/input/datasets/phmnmendis/olap-dataset-dnn-03/final_combined_olap.csv')

print(f'Data shape: {df.shape}')

# ===================== DANN frac=0.3 PRODUCTION RUN =====================
# Train with DANN + 30% target labels, Spearman-based early stopping
# Save best model for downstream labeling
N_FOLDS = 5
N_EPOCHS = 100

os.makedirs('/kaggle/working/artifacts', exist_ok=True)
start = time.time()

print("\n" + "#"*70)
print("TRAINING: DANN with 30% target labels (Spearman-optimized)")
print("#"*70)

metrics, history_df, model, per_domain = train_and_eval(
    df,
    normalization_mode="per_engine",
    n_epochs=N_EPOCHS,
    batch_size=256,
    lr=5e-4,
    seed=SEED,
    device_str='cuda' if torch.cuda.is_available() else 'cpu',
    log_target=True,
    use_sampler=True,
    n_folds=N_FOLDS,
    labeled_target_frac=0.3,
    use_dann=True,
)

# Save best model
if model is not None:
    torch.save(
        {
            'state_dict': model.state_dict(),
            'config': {
                'labeled_target_frac': 0.3,
                'use_dann': True,
                'n_epochs': N_EPOCHS,
                'n_folds': N_FOLDS,
                'seed': SEED,
                'normalization_mode': 'per_engine',
                'early_stopping': 'spearman',
            },
            'metrics': {
                'pooled': metrics['pooled'],
                'per_domain': metrics['per_domain'],
                'best_epoch': metrics['best_epoch'],
                'train_minutes': metrics['train_minutes'],
            }
        },
        '/kaggle/working/artifacts/dann_frac03_best.pt'
    )
    print("\nModel saved to /kaggle/working/artifacts/dann_frac03_best.pt")

# Save history
history_df.to_csv('/kaggle/working/artifacts/history_dann_frac03.csv', index=False)

# Save full metrics
with open('/kaggle/working/artifacts/metrics_dann_frac03.json', 'w') as f:
    json.dump({
        'pooled': metrics['pooled'],
        'per_domain': metrics['per_domain'],
        'best_epoch': metrics['best_epoch'],
        'train_minutes': metrics['train_minutes'],
        'fold_metrics': [{
            'fold': fm['fold'],
            'pooled_mape': fm['pooled_mape'],
            'pooled_rmse': fm['pooled_rmse'],
            'pooled_spearman': fm['pooled_spearman'],
            'best_epoch': fm['best_epoch'],
            'train_minutes': fm['train_minutes'],
            'per_domain': fm['per_domain'],
        } for fm in metrics.get('fold_metrics', [])],
    }, f, indent=2, default=str)

total_min = (time.time() - start) / 60.0
p = metrics['pooled']
print(f"\n{'='*70}")
print(f"FINAL RESULTS (DANN frac=0.3, {N_FOLDS}-fold, {total_min:.1f} min)")
print(f"{'='*70}")
print(f"  Pooled MAPE:     {p['mape']:.2f}%")
print(f"  Pooled Spearman: {p['spearman']:.4f}")
print(f"  Best Epoch:      {metrics['best_epoch']}")

print(f"\nPer-domain:")
for d in ['0', '1', '2', '3']:
    if d in metrics['per_domain']:
        dm = metrics['per_domain'][d]
        print(f"  D{d}: MAPE={dm.get('mape_mean', float('nan')):.2f}% | Spearman={dm.get('spearman_mean', float('nan')):.4f}")

print(f"\nDONE - total time: {total_min:.1f} minutes ({total_min/60:.1f} hours)")

